# AfterMath — Model Training

In [ ]:
import sys
sys.path.insert(0, '..')  # so `utils` (repo root) is importable when cwd is notebooks/

import os
import yaml
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

from utils.dataset import PairedCropDataset
from utils.model import SiameseDamageNet
from utils.training import compute_class_weights_tensor, EarlyStopper, train_one_epoch, validate
from utils.xbd_labels import DAMAGE_CLASSES

config = yaml.safe_load(open('../config.yaml'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Note on training data volume

This machine has no CUDA GPU. A benchmark of a single siamese forward+backward pass (resnet18 backbone, batch_size=32, 224x224 crops) took ~3.65s on CPU — training on the full ~33,600-row train split (80% of the ~42,000 'train'-split rows) would take many hours per epoch, which isn't practical here. Per the project owner's decision, the training manifest is subsampled to a cap of ~3,000 rows per `damage_class` before the train/val split, keeping CPU training to roughly 1-2 hours while still producing a legitimate fine-tuning result. This is consistent with the design spec's own stated scale target of subsampling to a manageable size so training finishes in a reasonable time. The `training.epochs` (30) and `early_stopping_patience` (5) config values are left untouched — only the data volume is reduced, not the epoch budget.

In [ ]:
manifest_path = '../' + config['data']['manifest']
full_manifest = pd.read_csv(manifest_path)
train_manifest = full_manifest[full_manifest['split'] == 'train'].reset_index(drop=True)

# Subsample to a cap per damage_class for tractable CPU training time (~1-2h).
# Classes with fewer than the cap (e.g. 'destroyed') keep all their rows.
MAX_PER_CLASS = 3000
train_manifest = (
    train_manifest.groupby('damage_class', group_keys=False)[train_manifest.columns]
    .apply(lambda g: g.sample(n=min(len(g), MAX_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)
print('Subsampled train_manifest class counts:')
print(train_manifest['damage_class'].value_counts())

# further split train into train/val (80/20), stratified by damage_class
from sklearn.model_selection import train_test_split
train_rows, val_rows = train_test_split(
    train_manifest, test_size=0.2, stratify=train_manifest['damage_class'], random_state=42
)
train_rows.to_csv('../data/processed/manifest_train.csv', index=False)
val_rows.to_csv('../data/processed/manifest_val.csv', index=False)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = PairedCropDataset('../data/processed/manifest_train.csv', transform=transform)
val_dataset = PairedCropDataset('../data/processed/manifest_val.csv', transform=transform)

batch_size = config['training']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [ ]:
class_counts = train_rows['damage_class'].value_counts().to_dict()
class_weights = compute_class_weights_tensor(class_counts, DAMAGE_CLASSES).to(device)

model = SiameseDamageNet(num_classes=config['model']['num_classes'], pretrained=config['model']['pretrained']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
stopper = EarlyStopper(patience=config['training']['early_stopping_patience'])

In [ ]:
train_losses, val_losses = [], []
best_val_loss = float('inf')
os.makedirs('../models', exist_ok=True)

for epoch in range(config['training']['epochs']):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate(model, val_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f'epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../models/best.pt')

    if stopper.step(val_loss):
        print('Early stopping triggered')
        break

In [ ]:
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.savefig('../docs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()